# Lesson 24 — Neural-Network-Embedded MINLP

## Learning objectives

1. Encode a trained feedforward NN as algebraic constraints in an MINLP.
2. Distinguish ReLU big-M, full-space, and reduced-space formulations.
3. Use `discopt.nn` to embed a Keras / ONNX model.
4. Apply NN-MINLP to surrogate-based optimization.

## 1. Why embed a NN?

You have a trained NN that approximates an expensive physics simulation. You want to *optimize* over its inputs subject to other (algebraic) constraints. Three approaches:

- **Black-box** (Bayesian / surrogate-based): cheap iterations, no certificate.
- **Embed as constraint**: solver sees the NN as algebraic equations; with MINLP, you can certify global optimality of the surrogate.
- **Differentiable layer** (Lesson 26): NN is a step inside a larger optimization-aware pipeline.

NN-MINLP took off after {cite:p}`Fischetti2018` showed ReLU networks reduce to MILP, and {cite:p}`Anderson2020` introduced strong formulations. {cite:p}`Ceccon2022` (OMLT) is the modern toolkit; `discopt.nn` follows the same patterns.

## 2. ReLU big-M

Let $a = w^\top x + b$ be the pre-activation, with bounds $\underline a \le a \le \overline a$. The activation $z = \max(0, a)$ is encoded as:

$$
\begin{aligned}
z &\ge a \\
z &\ge 0 \\
z &\le a - \underline a (1 - y) \\
z &\le \overline a \, y \\
y &\in \{0, 1\}
\end{aligned}
$$

This formulation is meaningful **only when the bounds straddle zero**, $\underline a \le 0 \le \overline a$ — then $y$ encodes which side ($a \le 0$ vs $a \ge 0$) is active. If $\underline a \ge 0$ the unit is in its linear regime ($z = a$, no binary needed); if $\overline a \le 0$ it is fixed off ($z = 0$). A practical ReLU embedder *checks each unit* and emits the cheapest valid form. Tight $\underline a, \overline a$ via bound propagation are critical to the LP-relaxation gap {cite:p}`Anderson2020,Fischetti2018`.

## 3. Smooth activations (sigmoid, tanh)

For sigmoid $z = \sigma(a)$, the constraint is genuinely nonlinear. Two options:

- **Full-space**: keep $z = \sigma(a)$ as a constraint; solve as MINLP.
- **Reduced-space**: substitute the NN equations through, leaving a single nested expression. Smaller variable count but worse for spatial B&B.

{cite:p}`Fischetti2018,Ceccon2022`.

## 4. Bound propagation

Propagate input bounds layer by layer. Split $w$ into non-negative and non-positive parts $w^+ = \max(w, 0) \ge 0$ and $w^- = \max(-w, 0) \ge 0$ (so $w = w^+ - w^-$). Then for each pre-activation $a = w^\top x + b$ with $x \in [\underline x, \overline x]$:

$$\underline a = (w^+)^\top \underline x \;-\; (w^-)^\top \overline x + b, \quad \overline a = (w^+)^\top \overline x \;-\; (w^-)^\top \underline x + b.$$

For deep networks, FBBT alone gives loose bounds; OBBT layer-by-layer is tighter. CROWN, IBP, and SymBP are NN-specific bound-propagation methods {cite:p}`Anderson2020`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# Hand-coded ReLU big-M for a tiny 2-input, 2-hidden, 1-output network.
# (`discopt.nn` ships `NetworkDefinition` + `ReluBigMFormulation` /
# `FullSpaceFormulation` etc.; we build the constraints manually below so
# you can see the formulation, then Exercise 1 has you use the higher-level
# `discopt.nn` API.)
import numpy as np, discopt as do

np.random.seed(0)
W1 = np.random.normal(size=(2, 2)); b1 = np.zeros(2)
W2 = np.random.normal(size=(2, 1)); b2 = np.zeros(1)

# Pre-activation bound (straddles zero for x ∈ [-1, 1]).
m = do.Model("nn_opt")
x = m.continuous("x", shape=(2,), lb=-1, ub=1)
a1 = W1 @ x + b1                                   # pre-activation, layer 1
ub1 = float(np.abs(W1).sum(axis=1).max() + np.abs(b1).max())
z1 = m.continuous("z1", shape=(2,), lb=0, ub=ub1)  # ReLU output, layer 1
y1_int = m.binary("y1_int", shape=(2,))
for i in range(2):
    m.subject_to(z1[i] >= a1[i])
    m.subject_to(z1[i] <= a1[i] + ub1 * (1 - y1_int[i]))
    m.subject_to(z1[i] <= ub1 * y1_int[i])
y_out = (W2 @ z1 + b2)[0]
m.maximize(y_out)
r = m.solve()
print("status:", r.status, " input:", r.value(x), " output ≈", r.objective)


## 5. Surrogate-based optimization

If your NN approximates an expensive simulation, then:

1. Sample inputs, evaluate simulation, train NN.
2. Optimize via NN-MINLP to find a candidate input.
3. Evaluate the simulation at that candidate.
4. Add to training set, retrain (or fine-tune); iterate.

This is **Bayesian-flavoured but with MINLP** instead of Gaussian processes. Useful when the simulation is structured (e.g., reactor design where physics gives you a known input domain).

## References
{cite:p}`Fischetti2018,Anderson2020,Ceccon2022,Wang2022`.